# Bab 2: Probabilitas, Keputusan, dan Informasi

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab02/bayes_dbd.py": "import numpy as np\n\n# Angka ilustrasi (bukan data klinis): pasien demam di puskesmas saat musim hujan\nprior = 0.05            # p(DBD): 5 dari 100 pasien demam  #<1>\nsens = 0.90             # p(positif | DBD)\nspes = 0.95             # p(negatif | bukan DBD)\n\n# Teorema Bayes\np_pos = sens * prior + (1 - spes) * (1 - prior)        # aturan jumlah  #<2>\npost = sens * prior / p_pos                             # aturan kali / Bayes\nprint(f\"p(positif)         = {p_pos:.4f}\")\nprint(f\"p(DBD | positif)   = {post:.3f}\")\n\n# Periksa dengan simulasi: satu juta pasien  #<3>\nrng = np.random.default_rng(0)\ndbd = rng.random(1_000_000) < prior\npositif = np.where(dbd, rng.random(dbd.size) < sens, rng.random(dbd.size) > spes)\nprint(f\"simulasi: {positif.sum()} positif, {(dbd & positif).sum()} di antaranya DBD\"\n      f\" -> {(dbd & positif).sum() / positif.sum():.3f}\")\n",
"bab02/entropi_teks.py": "import numpy as np\nfrom collections import Counter\n\nteks = (\"Bahwa sesungguhnya kemerdekaan itu ialah hak segala bangsa dan oleh \"\n        \"sebab itu, maka penjajahan di atas dunia harus dihapuskan, karena tidak \"\n        \"sesuai dengan perikemanusiaan dan perikeadilan.\")   # Pembukaan UUD 1945  #<1>\nhuruf = [c for c in teks.lower() if c.isalpha()]\nn = Counter(huruf)\np = np.array(list(n.values())) / len(huruf)\n\nH = -np.sum(p * np.log2(p))                      # entropi (bit)  #<2>\nprint(f\"{len(huruf)} huruf, {len(n)} jenis huruf berbeda\")\nprint(f\"entropi distribusi huruf : {H:.2f} bit per huruf\")\nprint(f\"batas atas log2({len(n)})      : {np.log2(len(n)):.2f} bit (semua huruf sama sering)\")\nprint(\"lima huruf tersering    :\", \", \".join(f\"{h}={c}\" for h, c in n.most_common(5)))\n\n# Divergensi KL tidak simetris  #<3>\nP = np.array([0.7, 0.2, 0.1])                    # misal: hujan, berawan, cerah (Januari)\nQ = np.array([0.2, 0.3, 0.5])                    # (Agustus)\nKL = lambda a, b: np.sum(a * np.log2(a / b))\nprint(f\"KL(P||Q) = {KL(P, Q):.3f} bit,  KL(Q||P) = {KL(Q, P):.3f} bit\")\n",
"bab02/keputusan_biaya.py": "import numpy as np\nfrom scipy.stats import norm\n\n# Model ilustrasi: jumlah trombosit (ribu per mikroliter) pada pasien demam\nprior = 0.05\ndbd, lain = norm(100, 40), norm(240, 40)        # p(x | kelas)  #<1>\n\ndef posterior(x):                               # p(DBD | x)\n    a = dbd.pdf(x) * prior\n    return a / (a + lain.pdf(x) * (1 - prior))\n\nrng = np.random.default_rng(1)\nn = 200_000\ny = rng.random(n) < prior                       # True = DBD\nx = np.where(y, dbd.rvs(n, random_state=rng), lain.rvs(n, random_state=rng))\np = posterior(x)\n\nL_lewat, L_rujuk = 20, 1     # biaya: DBD terlewat vs rujukan yang tidak perlu  #<2>\ndef laporan(nama, rujuk, mask=None):\n    m = np.ones(n, bool) if mask is None else mask\n    salah = (rujuk != y)[m].mean()\n    lewat = (y & ~rujuk)[m].sum()\n    biaya = (L_lewat * (y & ~rujuk) + L_rujuk * (~y & rujuk))[m].mean()\n    print(f\"{nama:28s} salah={salah:6.2%}  DBD terlewat={lewat:5d}  biaya={biaya:.3f}\")\n\nlaporan(\"A: p > 0,5 (galat minimum)\", p > 0.5)\nlaporan(\"B: p > 1/21 (biaya minimum)\", p > L_rujuk / (L_rujuk + L_lewat))  #<3>\nyakin = np.maximum(p, 1 - p) >= 0.95                                       #<4>\nlaporan(\"C: B + tolak bila ragu\", p > 1 / 21, yakin)\nprint(f\"   ditolak (perlu tes NS1): {1 - yakin.mean():.1%} pasien\")\nfor t in [80, 120, 150, 180]:\n    print(f\"   trombosit {t:3d} -> p(DBD | x) = {posterior(t):.3f}\")\n",
"bab02/kutukan_dimensi.py": "import numpy as np\n\nrng = np.random.default_rng(0)\nprint(\"   D   rasio jarak (maks-min)/min   volume di kulit 5%\")\nfor D in [2, 10, 100, 1000]:\n    X = rng.uniform(0, 1, (500, D))          # 500 titik acak dalam kubus satuan\n    q = rng.uniform(0, 1, D)                 # satu titik kueri\n    d = np.linalg.norm(X - q, axis=1)\n    kontras = (d.max() - d.min()) / d.min()  #<1>\n    kulit = 1 - 0.95**D                      # fraksi volume dekat permukaan  #<2>\n    print(f\"{D:5d}   {kontras:26.2f}   {kulit:18.3f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab02')
print('folder kerja:', os.getcwd())

## Listing 2.1: Teorema Bayes untuk skrining DBD, diperiksa dengan simulasi

Berkas: `kode/bab02/bayes_dbd.py`

In [ ]:
import numpy as np

# Angka ilustrasi (bukan data klinis): pasien demam di puskesmas saat musim hujan
prior = 0.05            # p(DBD): 5 dari 100 pasien demam  # (1)
sens = 0.90             # p(positif | DBD)
spes = 0.95             # p(negatif | bukan DBD)

# Teorema Bayes
p_pos = sens * prior + (1 - spes) * (1 - prior)        # aturan jumlah  # (2)
post = sens * prior / p_pos                             # aturan kali / Bayes
print(f"p(positif)         = {p_pos:.4f}")
print(f"p(DBD | positif)   = {post:.3f}")

# Periksa dengan simulasi: satu juta pasien  # (3)
rng = np.random.default_rng(0)
dbd = rng.random(1_000_000) < prior
positif = np.where(dbd, rng.random(dbd.size) < sens, rng.random(dbd.size) > spes)
print(f"simulasi: {positif.sum()} positif, {(dbd & positif).sum()} di antaranya DBD"
      f" -> {(dbd & positif).sum() / positif.sum():.3f}")

## Listing 2.2: Dua gejala kutukan dimensi

Berkas: `kode/bab02/kutukan_dimensi.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
print("   D   rasio jarak (maks-min)/min   volume di kulit 5%")
for D in [2, 10, 100, 1000]:
    X = rng.uniform(0, 1, (500, D))          # 500 titik acak dalam kubus satuan
    q = rng.uniform(0, 1, D)                 # satu titik kueri
    d = np.linalg.norm(X - q, axis=1)
    kontras = (d.max() - d.min()) / d.min()  # (1)
    kulit = 1 - 0.95**D                      # fraksi volume dekat permukaan  # (2)
    print(f"{D:5d}   {kontras:26.2f}   {kulit:18.3f}")

## Listing 2.3: Tiga aturan keputusan untuk rujukan DBD

Berkas: `kode/bab02/keputusan_biaya.py`

In [ ]:
import numpy as np
from scipy.stats import norm

# Model ilustrasi: jumlah trombosit (ribu per mikroliter) pada pasien demam
prior = 0.05
dbd, lain = norm(100, 40), norm(240, 40)        # p(x | kelas)  # (1)

def posterior(x):                               # p(DBD | x)
    a = dbd.pdf(x) * prior
    return a / (a + lain.pdf(x) * (1 - prior))

rng = np.random.default_rng(1)
n = 200_000
y = rng.random(n) < prior                       # True = DBD
x = np.where(y, dbd.rvs(n, random_state=rng), lain.rvs(n, random_state=rng))
p = posterior(x)

L_lewat, L_rujuk = 20, 1     # biaya: DBD terlewat vs rujukan yang tidak perlu  # (2)
def laporan(nama, rujuk, mask=None):
    m = np.ones(n, bool) if mask is None else mask
    salah = (rujuk != y)[m].mean()
    lewat = (y & ~rujuk)[m].sum()
    biaya = (L_lewat * (y & ~rujuk) + L_rujuk * (~y & rujuk))[m].mean()
    print(f"{nama:28s} salah={salah:6.2%}  DBD terlewat={lewat:5d}  biaya={biaya:.3f}")

laporan("A: p > 0,5 (galat minimum)", p > 0.5)
laporan("B: p > 1/21 (biaya minimum)", p > L_rujuk / (L_rujuk + L_lewat))  # (3)
yakin = np.maximum(p, 1 - p) >= 0.95                                       # (4)
laporan("C: B + tolak bila ragu", p > 1 / 21, yakin)
print(f"   ditolak (perlu tes NS1): {1 - yakin.mean():.1%} pasien")
for t in [80, 120, 150, 180]:
    print(f"   trombosit {t:3d} -> p(DBD | x) = {posterior(t):.3f}")

## Listing 2.4: Entropi distribusi huruf dan divergensi KL

Berkas: `kode/bab02/entropi_teks.py`

In [ ]:
import numpy as np
from collections import Counter

teks = ("Bahwa sesungguhnya kemerdekaan itu ialah hak segala bangsa dan oleh "
        "sebab itu, maka penjajahan di atas dunia harus dihapuskan, karena tidak "
        "sesuai dengan perikemanusiaan dan perikeadilan.")   # Pembukaan UUD 1945  # (1)
huruf = [c for c in teks.lower() if c.isalpha()]
n = Counter(huruf)
p = np.array(list(n.values())) / len(huruf)

H = -np.sum(p * np.log2(p))                      # entropi (bit)  # (2)
print(f"{len(huruf)} huruf, {len(n)} jenis huruf berbeda")
print(f"entropi distribusi huruf : {H:.2f} bit per huruf")
print(f"batas atas log2({len(n)})      : {np.log2(len(n)):.2f} bit (semua huruf sama sering)")
print("lima huruf tersering    :", ", ".join(f"{h}={c}" for h, c in n.most_common(5)))

# Divergensi KL tidak simetris  # (3)
P = np.array([0.7, 0.2, 0.1])                    # misal: hujan, berawan, cerah (Januari)
Q = np.array([0.2, 0.3, 0.5])                    # (Agustus)
KL = lambda a, b: np.sum(a * np.log2(a / b))
print(f"KL(P||Q) = {KL(P, Q):.3f} bit,  KL(Q||P) = {KL(Q, P):.3f} bit")